In [ ]:
import os
import pickle

import lmdb
import numpy as np
from matplotlib import pyplot as plt

# Visualize Sentinel-2 samples stored in LMDB.
# Layout mirrors visualize_samples.ipynb — one row per sample, one column per timestep.
# Use this to verify that LMDB creation does not alter the samples.

lmdb_path = os.environ.get(
    "S2_LMDB_PATH",
    "../../../sentinel/data/ssl4eo_consecutive_s2c.lmdb",
)

n_samples = 10
n_timesteps = 5
seed = None  # set to an int for reproducibility
TARGET_T = n_timesteps // 2

# Close a previously opened environment (important when re-running the cell).
try:
    if "env" in globals() and env is not None:
        env.close()
except Exception:
    pass

env = lmdb.open(
    lmdb_path,
    readonly=True,
    lock=False,
    readahead=False,
    meminit=False,
    subdir=False,
)

n_total = env.stat()["entries"]
if n_total == 0:
    raise RuntimeError(f"LMDB is empty: {lmdb_path}")


def _decode_value(data: bytes) -> np.ndarray:
    try:
        s2c_bytes, s2c_shape = pickle.loads(data)
        return np.frombuffer(s2c_bytes, dtype=np.float32).reshape(s2c_shape)
    except Exception:
        ndim = int(np.frombuffer(data, dtype=np.int32, count=1, offset=0)[0])
        shape = tuple(np.frombuffer(data, dtype=np.int32, count=ndim, offset=4).tolist())
        header_nbytes = 4 * (1 + ndim)
        return np.frombuffer(data, dtype=np.float32, offset=header_nbytes).reshape(shape)


def load_sample(idx: int) -> np.ndarray:
    with env.begin(write=False) as txn:
        data = txn.get(str(idx).encode())
    if data is None:
        raise IndexError(f"No entry for index {idx}")
    return _decode_value(data)


def robust_rgb(b4: np.ndarray, b3: np.ndarray, b2: np.ndarray,
               pmin: float = 2, pmax: float = 98,
               lo: float = None, hi: float = None):
    rgb = np.stack([b4, b3, b2], axis=-1).astype(np.float32)
    gaps = np.any(np.isnan(rgb), axis=-1)
    if lo is None or hi is None:
        lo = np.nanpercentile(rgb, pmin)
        hi = np.nanpercentile(rgb, pmax)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = 0.0, 2500.0
    rgb = (rgb - lo) / (hi - lo + 1e-6)
    rgb = np.clip(rgb, 0, 1)
    rgb[gaps] = 0.0
    return rgb, gaps


# Reservoir-sample n_samples indices from [0, n_total)
rng = np.random.default_rng(seed)
chosen_indices = []
for i in range(n_total):
    if len(chosen_indices) < n_samples:
        chosen_indices.append(i)
    else:
        j = int(rng.integers(0, i + 1))
        if j < n_samples:
            chosen_indices[j] = i
chosen_indices = sorted(chosen_indices)

fig, axes = plt.subplots(
    len(chosen_indices), n_timesteps, figsize=(20, 4 * len(chosen_indices))
)
if len(chosen_indices) == 1:
    axes = [axes]

for i, idx in enumerate(chosen_indices):
    s2c = load_sample(idx)  # (T, C, H, W)
    T, C, H, W = s2c.shape

    # Compute shared lo/hi from all timesteps so colors are consistent within the row
    all_raw = []
    for t in range(T):
        for arr in [s2c[t][2], s2c[t][1], s2c[t][0]]:  # B4, B3, B2
            valid = arr[~np.isnan(arr)]
            if valid.size:
                all_raw.append(valid)
    combined = np.concatenate(all_raw) if all_raw else np.array([0.0, 1.0])
    row_lo = float(np.percentile(combined, 2))
    row_hi = float(np.percentile(combined, 98))
    if not np.isfinite(row_lo) or not np.isfinite(row_hi) or row_hi <= row_lo:
        row_lo, row_hi = 0.0, 2500.0

    for t in range(n_timesteps):
        ax = axes[i][t]

        if t >= T:
            ax.imshow(np.zeros((10, 10, 3)))
            ax.set_title(f"Sample {i+1}, t={t}\n(missing)")
            ax.axis("off")
            continue

        # Band order from create_LMDB: sorted tifs → B2=ch0, B3=ch1, B4=ch2
        img = s2c[t]
        rgb, gaps = robust_rgb(img[2], img[1], img[0], lo=row_lo, hi=row_hi)

        ax.imshow(rgb)

        frac = 100.0 * float(gaps.sum()) / float(gaps.size)
        title = f"Sample {i+1} (idx={idx}), t={t}\n{H}×{W}  gaps={frac:.1f}%"
        if t == TARGET_T:
            title += " (target)"
        ax.set_title(title)
        ax.axis("off")

plt.tight_layout()
plt.show()